# 📖 Hadist API - Semantic Search (Vector Search) di Google Colab

Notebook ini memandu Anda untuk menjalankan pencarian semantik (Vector Search) hadis menggunakan **LanceDB** dan model **BAAI/bge-m3** di **Google Colab** dengan akselerasi **T4 GPU**.

--- 

### ⚡ 1. Pastikan GPU Aktif
Buka menu Colab: **Runtime** -> **Change runtime type** -> Pilih **T4 GPU**.

In [ ]:
# Cek ketersediaan GPU
!nvidia-smi

### 📥 2. Clone Repositori & Install Dependensi

In [ ]:
!git clone https://github.com/aarestu/hadist-api.git
%cd hadist-api
!pip install -r requirements.txt

### 💾 3. Impor Data Hadis ke Database Relasional
Menjelajah dan mengunduh data hadis dari API fawazahmed0/hadith-api.

In [ ]:
!python -m app.cli.main

### ⚡ 4. Tahap 1: Pembuatan Vektor Embeddings (Build Vectors di GPU)
Proses ini berjalan di atas GPU T4 dan menyimpannya secara incremental (*checkpointed*).

In [ ]:
!python -m app.cli.vector_cli build-vectors --batch-size 500

### 🚀 5. Tahap 2: Pembuatan LanceDB Index (Instant Build)
Membangun struktur index pencarian cepat tanpa kalkulasi ulang model.

In [ ]:
!python -m app.cli.vector_cli create-index

### 🔍 6. Uji Pencarian Semantik via CLI

In [ ]:
!python -m app.cli.vector_cli search -q "memuliakan dan menghormati tetangga" -k 5

### 🐍 7. Penggunaan Programatis via Kode Python di Colab

In [ ]:
from app.infrastructure.config import load_config
from app.services.vector_search_service import HadithVectorSearchService

config = load_config()
service = HadithVectorSearchService(config.vector_search)

# Kueri semantik secara langsung dari kode Python
query = "amalan tergantung niat"
results = await service.search(query=query, limit=3)

for r in results:
    print(f"Kitab: {r['book_name']} #{r['hadith_number']} | Skor: {r['score']*100:.1f}%")
    print(f"Teks: {r['indonesian_text'][:150]}...\n")